In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
import re
import snowflake.snowpark.functions as F
from snowflake.snowpark.functions import col, lit, regexp_replace, when

# SQL UDFs for core string manipulation functions
def register_udfs(session):
    print("Registering JavaScript UDFs for title variant generation...")
    
    # Register the JavaScript UDFs
    session.sql("""
    CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.IS_ARTICLE(word STRING)
    RETURNS BOOLEAN
    LANGUAGE JAVASCRIPT
    AS $$
        const articles = ['A', 'AN', 'THE'];
        return articles.includes(WORD.toUpperCase());
    $$;
    """).collect()
    
    session.sql("""
    CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.REMOVE_ARTICLES(title STRING)
    RETURNS STRING
    LANGUAGE JAVASCRIPT
    AS $$
        if (!TITLE) return '';
        
        const articles = ['A', 'AN', 'THE'];
        const words = TITLE.trim().split(/\s+/);
        
        // Remove leading article
        if (words.length > 0 && articles.includes(words[0].toUpperCase())) {
            words.shift();
        }
        
        // Remove trailing article
        if (words.length > 0 && articles.includes(words[words.length - 1].toUpperCase())) {
            words.pop();
        }
        
        return words.join(' ');
    $$;
    """).collect()
    
    session.sql("""
    CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.NORMALIZE_WHITESPACE(title STRING)
    RETURNS STRING
    LANGUAGE JAVASCRIPT
    AS $$
        if (!TITLE) return '';
        return TITLE.trim().replace(/\s+/g, ' ');
    $$;
    """).collect()
    
    session.sql("""
    CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.CLEAN_UNMATCHED_BRACKETS(title STRING)
    RETURNS STRING
    LANGUAGE JAVASCRIPT
    AS $$
        if (!TITLE) return '';
        
        const stack = [];
        const bracketPairs = {'(': ')', '[': ']', '{': '}'};
        const reversePairs = {')': '(', ']': '[', '}': '{'};
        const skipIndices = new Set();
        
        // First pass - identify unmatched brackets
        for (let i = 0; i < TITLE.length; i++) {
            const char = TITLE[i];
            
            if (bracketPairs[char]) {  // Opening bracket
                stack.push([char, i]);
            } else if (reversePairs[char]) {  // Closing bracket
                if (stack.length && stack[stack.length - 1][0] === reversePairs[char]) {
                    stack.pop();  // Matched pair
                } else {
                    // Unmatched closing bracket - mark for removal
                    skipIndices.add(i);
                }
            }
        }
        
        // Any remaining opening brackets are unmatched
        for (const [_, idx] of stack) {
            skipIndices.add(idx);
        }
        
        // Build cleaned title without unmatched brackets
        let cleanedTitle = "";
        for (let i = 0; i < TITLE.length; i++) {
            if (!skipIndices.has(i)) {
                cleanedTitle += TITLE[i];
            }
        }
        
        return cleanedTitle.trim().replace(/\s+/g, ' ');
    $$;
    """).collect()
    
    # Complex variant generation function
    # This is a simplified JavaScript version that returns JSON with variants
    session.sql("""
    CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.GENERATE_TITLE_VARIANTS_JS(title STRING)
    RETURNS VARIANT
    LANGUAGE JAVASCRIPT
    AS $$
        // Handle NULL titles
        if (TITLE === null || TITLE === undefined) {
            return [];
        }
        
        const originalTitle = TITLE.trim();
        
        // Helper functions
        function isArticle(word) {
            const articles = ['A', 'AN', 'THE'];
            return articles.includes(word.toUpperCase());
        }
        
        function removeArticles(title) {
            if (!title) return '';
            const words = title.trim().split(/\s+/);
            
            if (words.length > 0 && isArticle(words[0])) {
                words.shift();
            }
            
            if (words.length > 0 && isArticle(words[words.length - 1])) {
                words.pop();
            }
            
            return words.join(' ');
        }
        
        function normalizeWhitespace(title) {
            if (!title) return '';
            return title.trim().replace(/\s+/g, ' ');
        }
        
        function cleanUnmatchedBrackets(title) {
            if (!title) return '';
            
            const stack = [];
            const bracketPairs = {'(': ')', '[': ']', '{': '}'};
            const reversePairs = {')': '(', ']': '[', '}': '{'};
            const skipIndices = new Set();
            
            // First pass - identify unmatched brackets
            for (let i = 0; i < title.length; i++) {
                const char = title[i];
                
                if (bracketPairs[char]) {  // Opening bracket
                    stack.push([char, i]);
                } else if (reversePairs[char]) {  // Closing bracket
                    if (stack.length && stack[stack.length - 1][0] === reversePairs[char]) {
                        stack.pop();  // Matched pair
                    } else {
                        // Unmatched closing bracket - mark for removal
                        skipIndices.add(i);
                    }
                }
            }
            
            // Any remaining opening brackets are unmatched
            for (const [_, idx] of stack) {
                skipIndices.add(idx);
            }
            
            // Build cleaned title without unmatched brackets
            let cleanedTitle = "";
            for (let i = 0; i < title.length; i++) {
                if (!skipIndices.has(i)) {
                    cleanedTitle += title[i];
                }
            }
            
            return normalizeWhitespace(cleanedTitle);
        }
        
        // Main variant generation logic
        const cleanedTitle = cleanUnmatchedBrackets(originalTitle);
        const variants = new Set();
        
        // FIXED: Store only the regex pattern source, not the regex object itself
        const bracketPatternSource = /([\[\(\{])([^\[\]\(\)\{\}]*)([\]\)\}])/g.source;
        
        // Find all matched brackets in the cleaned title
        const bracketMatches = [];
        let match;
        // FIXED: Always create a new RegExp instance when using the pattern
        const patternForMatching = new RegExp(bracketPatternSource, 'g');
        while ((match = patternForMatching.exec(cleanedTitle)) !== null) {
            bracketMatches.push({
                full: match[0],
                open: match[1],
                content: match[2],
                close: match[3],
                start: match.index,
                end: match.index + match[0].length
            });
        }
        
        // If no brackets, check if original had any
        if (bracketMatches.length === 0) {
            if (/[\[\]\(\)\{\}]/.test(originalTitle)) {
                const cleanTitle = removeArticles(cleanedTitle);
                if (cleanTitle) {
                    variants.add(cleanTitle);
                }
            }
            return Array.from(variants).sort();
        }
        
        // FIXED: Create a new RegExp instance for each replacement
        const noBrackets = cleanedTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
        const normalizedNoBrackets = normalizeWhitespace(noBrackets);
        const noArticlesNoBrackets = removeArticles(normalizedNoBrackets);
        
        if (noArticlesNoBrackets && 
            noArticlesNoBrackets.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
            variants.add(noArticlesNoBrackets);
        }
        
        // Group adjacent brackets
        const bracketGroups = [];
        if (bracketMatches.length > 0) {
            let currentGroup = [bracketMatches[0]];
            
            for (let i = 1; i < bracketMatches.length; i++) {
                const prevEnd = bracketMatches[i-1].end;
                const currentStart = bracketMatches[i].start;
                
                // Check if there's text between brackets
                if (cleanedTitle.substring(prevEnd, currentStart).trim()) {
                    // Non-adjacent brackets, start a new group
                    bracketGroups.push({
                        start: currentGroup[0].start,
                        end: currentGroup[currentGroup.length-1].end,
                        matches: [...currentGroup]
                    });
                    currentGroup = [bracketMatches[i]];
                } else {
                    // Adjacent brackets, add to current group
                    currentGroup.push(bracketMatches[i]);
                }
            }
            
            // Add the last group
            bracketGroups.push({
                start: currentGroup[0].start,
                end: currentGroup[currentGroup.length-1].end,
                matches: [...currentGroup]
            });
        }
        
        // Split the title into segments: text segments and bracket groups
        const segments = [];
        let lastEnd = 0;
        
        for (const group of bracketGroups) {
            // Add text before this group
            if (group.start > lastEnd) {
                segments.push({
                    type: 'text',
                    content: cleanedTitle.substring(lastEnd, group.start)
                });
            }
            
            // Add this bracket group
            segments.push({
                type: 'group',
                content: group.matches
            });
            
            // Update lastEnd
            lastEnd = group.end;
        }
        
        // Add any remaining text after the last bracket group
        if (lastEnd < cleanedTitle.length) {
            segments.push({
                type: 'text',
                content: cleanedTitle.substring(lastEnd)
            });
        }
        
        // Variant 1: with first bracket of first group only
        const firstVariantParts = [];
        
        for (const segment of segments) {
            if (segment.type === 'text') {
                firstVariantParts.push(segment.content);
            } else if (segment.type === 'group') {
                // Add only first bracket content from first group
                firstVariantParts.push(segment.content[0].content);
            }
        }
        
        const firstVariant = removeArticles(normalizeWhitespace(firstVariantParts.join(' ')));
        
        if (firstVariant && 
            !variants.has(firstVariant) && 
            firstVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
            variants.add(firstVariant);
        }
        
        // Variant 2: If multiple groups, create variant with first bracket of first group
        // and first bracket of last group
        if (bracketGroups.length > 1) {
            const secondVariantParts = [];
            let firstGroupUsed = false;
            let lastGroupUsed = false;
            
            for (const segment of segments) {
                if (segment.type === 'text') {
                    secondVariantParts.push(segment.content);
                } else if (segment.type === 'group') {
                    // Check if this is first group
                    if (!firstGroupUsed && segment.content[0].start === bracketGroups[0].matches[0].start) {
                        secondVariantParts.push(segment.content[0].content);
                        firstGroupUsed = true;
                    } 
                    // Check if this is last group
                    else if (!lastGroupUsed && 
                            segment.content[0].start === bracketGroups[bracketGroups.length-1].matches[0].start &&
                            bracketGroups[0].start !== bracketGroups[bracketGroups.length-1].start) {
                        secondVariantParts.push(segment.content[0].content);
                        lastGroupUsed = true;
                    }
                }
            }
            
            const secondVariant = removeArticles(normalizeWhitespace(secondVariantParts.join(' ')));
            
            if (secondVariant && 
                !variants.has(secondVariant) && 
                secondVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
                variants.add(secondVariant);
            }
        }
        
        // Variant 3: If multiple groups, create variant with just text and first bracket of last group
        if (bracketGroups.length > 1) {
            const thirdVariantParts = [];
            let lastGroupUsed = false;
            
            for (const segment of segments) {
                if (segment.type === 'text') {
                    thirdVariantParts.push(segment.content);
                } else if (segment.type === 'group' && !lastGroupUsed && 
                        segment.content[0].start === bracketGroups[bracketGroups.length-1].matches[0].start) {
                    thirdVariantParts.push(segment.content[0].content);
                    lastGroupUsed = true;
                }
            }
            
            const thirdVariant = removeArticles(normalizeWhitespace(thirdVariantParts.join(' ')));
            
            if (thirdVariant && 
                !variants.has(thirdVariant) && 
                thirdVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
                variants.add(thirdVariant);
            }
        }
        
        // Variant 4: Handle titles that start with a bracket
        if (bracketMatches.length > 0 && bracketMatches[0].start === 0) {
            // Get just the content of the first bracket
            const bracketContent = normalizeWhitespace(bracketMatches[0].content);
            let restOfTitle = cleanedTitle.substring(bracketMatches[0].end);
            
            // FIXED: Use a new RegExp instance for replacement
            restOfTitle = restOfTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
            restOfTitle = normalizeWhitespace(restOfTitle);
            
            // Create variant with just bracket content + rest of title
            const combined = removeArticles(normalizeWhitespace(`${bracketContent} ${restOfTitle}`));
            
            if (combined && 
                !variants.has(combined) && 
                combined.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
                variants.add(combined);
            }
            
            // Also create a variant with just the bracket content if it's followed by more brackets
            if (bracketMatches.length > 1 && bracketMatches[1].start === bracketMatches[0].end) {
                const justContent = removeArticles(bracketContent);
                
                if (justContent && 
                    !variants.has(justContent) && 
                    justContent.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
                    variants.add(justContent);
                }
            }
        }
        
        // Complex case handling for titles like "(NEW WORLD) (HI) HELLO (SMILE TIME) WORLD (TIME) (XXXX)"
        if (bracketGroups.length >= 2) {
            // Find brackets that aren't in any group (middle brackets)
            const middleBrackets = [];
            
            for (const match of bracketMatches) {
                let isInGroup = false;
                
                for (const group of bracketGroups) {
                    for (const groupMatch of group.matches) {
                        if (match.start === groupMatch.start && match.end === groupMatch.end) {
                            isInGroup = true;
                            break;
                        }
                    }
                    
                    if (isInGroup) break;
                }
                
                if (!isInGroup) {
                    middleBrackets.push(match);
                }
            }
            
            // If we have middle brackets between groups
            if (middleBrackets.length > 0) {
                // Create variant with first group content + middle brackets + last group content
                const complexVariantParts = [];
                let baseTextAdded = false;
                
                // Add first group content
                complexVariantParts.push(bracketGroups[0].matches[0].content);
                
                // FIXED: Use a new RegExp instance for replacement
                const baseText = cleanedTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
                complexVariantParts.push(baseText);
                baseTextAdded = true;
                
                // Add content from middle brackets
                for (const middleMatch of middleBrackets) {
                    complexVariantParts.push(middleMatch.content);
                }
                
                // Add last group content (first bracket only)
                complexVariantParts.push(bracketGroups[bracketGroups.length-1].matches[0].content);
                
                const complexVariant = removeArticles(normalizeWhitespace(complexVariantParts.join(' ')));
                
                if (complexVariant && 
                    !variants.has(complexVariant) && 
                    complexVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
                    variants.add(complexVariant);
                }
                
                // Also create variant with base text + last group content
                if (!baseTextAdded) {
                    const baseVariantParts = [
                        baseText, 
                        bracketGroups[bracketGroups.length-1].matches[0].content
                    ];
                    
                    const baseVariant = removeArticles(normalizeWhitespace(baseVariantParts.join(' ')));
                    
                    if (baseVariant && 
                        !variants.has(baseVariant) && 
                        baseVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
                        variants.add(baseVariant);
                    }
                }
            }
        }
        
        return Array.from(variants).sort();
    $$;           
    """).collect()
    
    return


def process_batches_sql(session, table_name):
    """Process title variants using SQL and the registered JavaScript UDFs"""
    print("Processing title variants using SQL...")
    
    # Create a temporary table to store results
    session.sql(f"""
    CREATE OR REPLACE TEMPORARY TABLE EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS AS
    SELECT
        t.*,
        t.TITLE as ORIGINAL_TITLE,
        NULL as TITLE_VARIANT,
        FALSE as IS_VARIANT
    FROM {table_name} t
    """).collect()
    
    # Process variants using the JavaScript UDF and insert into the temporary table
    session.sql(f"""
    INSERT INTO EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS
    SELECT 
        t.*,
        t.TITLE as ORIGINAL_TITLE,
        v.value::STRING as TITLE_VARIANT,
        TRUE as IS_VARIANT
    FROM {table_name} t,
    TABLE(FLATTEN(EDW_APPS.MATCHING.GENERATE_TITLE_VARIANTS_JS(t.TITLE))) v
    WHERE v.value IS NOT NULL
    """).collect()
    
    # Update original rows to include their TITLE as TITLE_VARIANT if they have no variants
    session.sql("""
    UPDATE EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS orig
    SET TITLE_VARIANT = TITLE
    WHERE IS_VARIANT = FALSE
    AND NOT EXISTS (
        SELECT 1 FROM EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS var
        WHERE var.ORIGINAL_TITLE = orig.TITLE
        AND var.IS_VARIANT = TRUE
    )
    """).collect()
    
    # Create the final table
    session.sql("""
    CREATE OR REPLACE TABLE EDW_APPS.MATCHING.ADC_WORKS_TITLE_VARIANTS AS
    SELECT * FROM EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS
    //WHERE TITLE_VARIANT IS NOT NULL
    """).collect()
    
    stats = session.sql("""
    SELECT
        SUM(CASE WHEN IS_VARIANT = FALSE THEN 1 ELSE 0 END) as ORIGINAL_COUNT,
        SUM(CASE WHEN IS_VARIANT = TRUE THEN 1 ELSE 0 END) as VARIANT_COUNT,
        COUNT(*) as TOTAL_COUNT
    FROM EDW_APPS.MATCHING.ADC_WORKS_TITLE_VARIANTS
    """).collect()
    
    print(f"OAriginal tracks: {stats[0]['ORIGINAL_COUNT']}")
    print(f"Generated variants: {stats[0]['VARIANT_COUNT']}")
    print(f"Total rows in output: {stats[0]['TOTAL_COUNT']}")
    
    return


def main(session):
    # Get the input data from ADCWORKS
    print("Reading data from ADCWORKS...")
    table_name = "EDW_APPS.MATCHING.ADCWORKS"
    
    # Register JavaScript UDFs for title variant generation
    register_udfs(session)
    
    # Process data in SQL batches
    process_batches_sql(session, table_name)

    
    print("\nTitle variant processing complete!")
    return session.table("EDW_APPS.MATCHING.ADC_WORKS_TITLE_VARIANTS")

# Run the main function
result_df = main(session)

In [ ]:
# import re
# import snowflake.snowpark.functions as F
# from snowflake.snowpark.functions import col, lit, regexp_replace, when

# # Register JavaScript UDFs for title variant generation
# def register_udfs(session):
#     print("Registering JavaScript UDFs for title variant generation...")
    
#     # Register the JavaScript UDFs
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.IS_ARTICLE(word STRING)
#     RETURNS BOOLEAN
#     LANGUAGE JAVASCRIPT
#     AS $$
#         const articles = ['A', 'AN', 'THE'];
#         return articles.includes(WORD.toUpperCase());
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.REMOVE_ARTICLES(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
        
#         const articles = ['A', 'AN', 'THE'];
#         const words = TITLE.trim().split(/\s+/);
        
#         // Remove leading article
#         if (words.length > 0 && articles.includes(words[0].toUpperCase())) {
#             words.shift();
#         }
        
#         // Remove trailing article
#         if (words.length > 0 && articles.includes(words[words.length - 1].toUpperCase())) {
#             words.pop();
#         }
        
#         return words.join(' ');
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.NORMALIZE_WHITESPACE(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
#         return TITLE.trim().replace(/\s+/g, ' ');
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.CLEAN_UNMATCHED_BRACKETS(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
        
#         const stack = [];
#         const bracketPairs = {'(': ')', '[': ']', '{': '}'};
#         const reversePairs = {')': '(', ']': '[', '}': '{'};
#         const skipIndices = new Set();
        
#         // First pass - identify unmatched brackets
#         for (let i = 0; i < TITLE.length; i++) {
#             const char = TITLE[i];
            
#             if (bracketPairs[char]) {  // Opening bracket
#                 stack.push([char, i]);
#             } else if (reversePairs[char]) {  // Closing bracket
#                 if (stack.length && stack[stack.length - 1][0] === reversePairs[char]) {
#                     stack.pop();  // Matched pair
#                 } else {
#                     // Unmatched closing bracket - mark for removal
#                     skipIndices.add(i);
#                 }
#             }
#         }
        
#         // Any remaining opening brackets are unmatched
#         for (const [_, idx] of stack) {
#             skipIndices.add(idx);
#         }
        
#         // Build cleaned title without unmatched brackets
#         let cleanedTitle = "";
#         for (let i = 0; i < TITLE.length; i++) {
#             if (!skipIndices.has(i)) {
#                 cleanedTitle += TITLE[i];
#             }
#         }
        
#         return cleanedTitle.trim().replace(/\s+/g, ' ');
#     $$;
#     """).collect()
    
#     # Complex variant generation function that handles NULL titles
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.GENERATE_TITLE_VARIANTS_JS(title STRING)
#     RETURNS VARIANT
#     LANGUAGE JAVASCRIPT
#     AS $$
#         // Handle NULL titles
#         if (TITLE === null || TITLE === undefined) {
#             return [];
#         }
        
#         const originalTitle = TITLE.trim();
        
#         // Helper functions
#         function isArticle(word) {
#             const articles = ['A', 'AN', 'THE'];
#             return articles.includes(word.toUpperCase());
#         }
        
#         function removeArticles(title) {
#             if (!title) return '';
#             const words = title.trim().split(/\s+/);
            
#             if (words.length > 0 && isArticle(words[0])) {
#                 words.shift();
#             }
            
#             if (words.length > 0 && isArticle(words[words.length - 1])) {
#                 words.pop();
#             }
            
#             return words.join(' ');
#         }
        
#         function normalizeWhitespace(title) {
#             if (!title) return '';
#             return title.trim().replace(/\s+/g, ' ');
#         }
        
#         function cleanUnmatchedBrackets(title) {
#             if (!title) return '';
            
#             const stack = [];
#             const bracketPairs = {'(': ')', '[': ']', '{': '}'};
#             const reversePairs = {')': '(', ']': '[', '}': '{'};
#             const skipIndices = new Set();
            
#             // First pass - identify unmatched brackets
#             for (let i = 0; i < title.length; i++) {
#                 const char = title[i];
                
#                 if (bracketPairs[char]) {  // Opening bracket
#                     stack.push([char, i]);
#                 } else if (reversePairs[char]) {  // Closing bracket
#                     if (stack.length && stack[stack.length - 1][0] === reversePairs[char]) {
#                         stack.pop();  // Matched pair
#                     } else {
#                         // Unmatched closing bracket - mark for removal
#                         skipIndices.add(i);
#                     }
#                 }
#             }
            
#             // Any remaining opening brackets are unmatched
#             for (const [_, idx] of stack) {
#                 skipIndices.add(idx);
#             }
            
#             // Build cleaned title without unmatched brackets
#             let cleanedTitle = "";
#             for (let i = 0; i < title.length; i++) {
#                 if (!skipIndices.has(i)) {
#                     cleanedTitle += title[i];
#                 }
#             }
            
#             return normalizeWhitespace(cleanedTitle);
#         }
        
#         // Main variant generation logic
#         const cleanedTitle = cleanUnmatchedBrackets(originalTitle);
#         const variants = new Set();
        
#         // FIXED: Store only the regex pattern source, not the regex object itself
#         const bracketPatternSource = /([\[\(\{])([^\[\]\(\)\{\}]*)([\]\)\}])/g.source;
        
#         // Find all matched brackets in the cleaned title
#         const bracketMatches = [];
#         let match;
#         // FIXED: Always create a new RegExp instance when using the pattern
#         const patternForMatching = new RegExp(bracketPatternSource, 'g');
#         while ((match = patternForMatching.exec(cleanedTitle)) !== null) {
#             bracketMatches.push({
#                 full: match[0],
#                 open: match[1],
#                 content: match[2],
#                 close: match[3],
#                 start: match.index,
#                 end: match.index + match[0].length
#             });
#         }
        
#         // If no brackets, check if original had any
#         if (bracketMatches.length === 0) {
#             if (/[\[\]\(\)\{\}]/.test(originalTitle)) {
#                 const cleanTitle = removeArticles(cleanedTitle);
#                 if (cleanTitle) {
#                     variants.add(cleanTitle);
#                 }
#             }
#             return Array.from(variants).sort();
#         }
        
#         // FIXED: Create a new RegExp instance for each replacement
#         const noBrackets = cleanedTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#         const normalizedNoBrackets = normalizeWhitespace(noBrackets);
#         const noArticlesNoBrackets = removeArticles(normalizedNoBrackets);
        
#         if (noArticlesNoBrackets && 
#             noArticlesNoBrackets.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#             variants.add(noArticlesNoBrackets);
#         }
        
#         // Group adjacent brackets
#         const bracketGroups = [];
#         if (bracketMatches.length > 0) {
#             let currentGroup = [bracketMatches[0]];
            
#             for (let i = 1; i < bracketMatches.length; i++) {
#                 const prevEnd = bracketMatches[i-1].end;
#                 const currentStart = bracketMatches[i].start;
                
#                 // Check if there's text between brackets
#                 if (cleanedTitle.substring(prevEnd, currentStart).trim()) {
#                     // Non-adjacent brackets, start a new group
#                     bracketGroups.push({
#                         start: currentGroup[0].start,
#                         end: currentGroup[currentGroup.length-1].end,
#                         matches: [...currentGroup]
#                     });
#                     currentGroup = [bracketMatches[i]];
#                 } else {
#                     // Adjacent brackets, add to current group
#                     currentGroup.push(bracketMatches[i]);
#                 }
#             }
            
#             // Add the last group
#             bracketGroups.push({
#                 start: currentGroup[0].start,
#                 end: currentGroup[currentGroup.length-1].end,
#                 matches: [...currentGroup]
#             });
#         }
        
#         // Split the title into segments: text segments and bracket groups
#         const segments = [];
#         let lastEnd = 0;
        
#         for (const group of bracketGroups) {
#             // Add text before this group
#             if (group.start > lastEnd) {
#                 segments.push({
#                     type: 'text',
#                     content: cleanedTitle.substring(lastEnd, group.start)
#                 });
#             }
            
#             // Add this bracket group
#             segments.push({
#                 type: 'group',
#                 content: group.matches
#             });
            
#             // Update lastEnd
#             lastEnd = group.end;
#         }
        
#         // Add any remaining text after the last bracket group
#         if (lastEnd < cleanedTitle.length) {
#             segments.push({
#                 type: 'text',
#                 content: cleanedTitle.substring(lastEnd)
#             });
#         }
        
#         // Variant 1: with first bracket of first group only
#         const firstVariantParts = [];
        
#         for (const segment of segments) {
#             if (segment.type === 'text') {
#                 firstVariantParts.push(segment.content);
#             } else if (segment.type === 'group') {
#                 // Add only first bracket content from first group
#                 firstVariantParts.push(segment.content[0].content);
#             }
#         }
        
#         const firstVariant = removeArticles(normalizeWhitespace(firstVariantParts.join(' ')));
        
#         if (firstVariant && 
#             !variants.has(firstVariant) && 
#             firstVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#             variants.add(firstVariant);
#         }
        
#         // Variant 2: If multiple groups, create variant with first bracket of first group
#         // and first bracket of last group
#         if (bracketGroups.length > 1) {
#             const secondVariantParts = [];
#             let firstGroupUsed = false;
#             let lastGroupUsed = false;
            
#             for (const segment of segments) {
#                 if (segment.type === 'text') {
#                     secondVariantParts.push(segment.content);
#                 } else if (segment.type === 'group') {
#                     // Check if this is first group
#                     if (!firstGroupUsed && segment.content[0].start === bracketGroups[0].matches[0].start) {
#                         secondVariantParts.push(segment.content[0].content);
#                         firstGroupUsed = true;
#                     } 
#                     // Check if this is last group
#                     else if (!lastGroupUsed && 
#                             segment.content[0].start === bracketGroups[bracketGroups.length-1].matches[0].start &&
#                             bracketGroups[0].start !== bracketGroups[bracketGroups.length-1].start) {
#                         secondVariantParts.push(segment.content[0].content);
#                         lastGroupUsed = true;
#                     }
#                 }
#             }
            
#             const secondVariant = removeArticles(normalizeWhitespace(secondVariantParts.join(' ')));
            
#             if (secondVariant && 
#                 !variants.has(secondVariant) && 
#                 secondVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(secondVariant);
#             }
#         }
        
#         // Variant 3: If multiple groups, create variant with just text and first bracket of last group
#         if (bracketGroups.length > 1) {
#             const thirdVariantParts = [];
#             let lastGroupUsed = false;
            
#             for (const segment of segments) {
#                 if (segment.type === 'text') {
#                     thirdVariantParts.push(segment.content);
#                 } else if (segment.type === 'group' && !lastGroupUsed && 
#                         segment.content[0].start === bracketGroups[bracketGroups.length-1].matches[0].start) {
#                     thirdVariantParts.push(segment.content[0].content);
#                     lastGroupUsed = true;
#                 }
#             }
            
#             const thirdVariant = removeArticles(normalizeWhitespace(thirdVariantParts.join(' ')));
            
#             if (thirdVariant && 
#                 !variants.has(thirdVariant) && 
#                 thirdVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(thirdVariant);
#             }
#         }
        
#         // Variant 4: Handle titles that start with a bracket
#         if (bracketMatches.length > 0 && bracketMatches[0].start === 0) {
#             // Get just the content of the first bracket
#             const bracketContent = normalizeWhitespace(bracketMatches[0].content);
#             let restOfTitle = cleanedTitle.substring(bracketMatches[0].end);
            
#             // FIXED: Use a new RegExp instance for replacement
#             restOfTitle = restOfTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#             restOfTitle = normalizeWhitespace(restOfTitle);
            
#             // Create variant with just bracket content + rest of title
#             const combined = removeArticles(normalizeWhitespace(`${bracketContent} ${restOfTitle}`));
            
#             if (combined && 
#                 !variants.has(combined) && 
#                 combined.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(combined);
#             }
            
#             // Also create a variant with just the bracket content if it's followed by more brackets
#             if (bracketMatches.length > 1 && bracketMatches[1].start === bracketMatches[0].end) {
#                 const justContent = removeArticles(bracketContent);
                
#                 if (justContent && 
#                     !variants.has(justContent) && 
#                     justContent.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                     variants.add(justContent);
#                 }
#             }
#         }
        
#         // Complex case handling for titles like "(NEW WORLD) (HI) HELLO (SMILE TIME) WORLD (TIME) (XXXX)"
#         if (bracketGroups.length >= 2) {
#             // Find brackets that aren't in any group (middle brackets)
#             const middleBrackets = [];
            
#             for (const match of bracketMatches) {
#                 let isInGroup = false;
                
#                 for (const group of bracketGroups) {
#                     for (const groupMatch of group.matches) {
#                         if (match.start === groupMatch.start && match.end === groupMatch.end) {
#                             isInGroup = true;
#                             break;
#                         }
#                     }
                    
#                     if (isInGroup) break;
#                 }
                
#                 if (!isInGroup) {
#                     middleBrackets.push(match);
#                 }
#             }
            
#             // If we have middle brackets between groups
#             if (middleBrackets.length > 0) {
#                 // Create variant with first group content + middle brackets + last group content
#                 const complexVariantParts = [];
#                 let baseTextAdded = false;
                
#                 // Add first group content
#                 complexVariantParts.push(bracketGroups[0].matches[0].content);
                
#                 // FIXED: Use a new RegExp instance for replacement
#                 const baseText = cleanedTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#                 complexVariantParts.push(baseText);
#                 baseTextAdded = true;
                
#                 // Add content from middle brackets
#                 for (const middleMatch of middleBrackets) {
#                     complexVariantParts.push(middleMatch.content);
#                 }
                
#                 // Add last group content (first bracket only)
#                 complexVariantParts.push(bracketGroups[bracketGroups.length-1].matches[0].content);
                
#                 const complexVariant = removeArticles(normalizeWhitespace(complexVariantParts.join(' ')));
                
#                 if (complexVariant && 
#                     !variants.has(complexVariant) && 
#                     complexVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                     variants.add(complexVariant);
#                 }
                
#                 // Also create variant with base text + last group content
#                 if (!baseTextAdded) {
#                     const baseVariantParts = [
#                         baseText, 
#                         bracketGroups[bracketGroups.length-1].matches[0].content
#                     ];
                    
#                     const baseVariant = removeArticles(normalizeWhitespace(baseVariantParts.join(' ')));
                    
#                     if (baseVariant && 
#                         !variants.has(baseVariant) && 
#                         baseVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                         variants.add(baseVariant);
#                     }
#                 }
#             }
#         }
        
#         return Array.from(variants).sort();
#     $$;           
#     """).collect()
    
#     return


# def process_batches_sql(session, table_name):
#     """Process title variants using SQL and the registered JavaScript UDFs"""
#     print("Processing title variants using SQL...")
    
#     # Create a temporary table to store results
#     session.sql(f"""
#     CREATE OR REPLACE TEMPORARY TABLE EDW_APPS.MATCHING.TEMP_MZK_TITLE_VARIANTS AS
#     SELECT
#         t.*,
#         t.TITLE as ORIGINAL_TITLE,
#         NULL as TITLE_VARIANT,
#         FALSE as IS_VARIANT
#     FROM {table_name} t
#     """).collect()
    
#     # Process variants using the JavaScript UDF and insert into the temporary table
#     session.sql(f"""
#     INSERT INTO EDW_APPS.MATCHING.TEMP_MZK_TITLE_VARIANTS
#     SELECT 
#         t.*,
#         t.TITLE as ORIGINAL_TITLE,
#         v.value::STRING as TITLE_VARIANT,
#         TRUE as IS_VARIANT
#     FROM {table_name} t,
#     TABLE(FLATTEN(EDW_APPS.MATCHING.GENERATE_TITLE_VARIANTS_JS(t.TITLE))) v
#     WHERE v.value IS NOT NULL
#     """).collect()
    
#     # Update original rows to include their TITLE as TITLE_VARIANT if they have no variants
#     session.sql("""
#     UPDATE EDW_APPS.MATCHING.TEMP_MZK_TITLE_VARIANTS orig
#     SET TITLE_VARIANT = TITLE
#     WHERE IS_VARIANT = FALSE
#     AND NOT EXISTS (
#         SELECT 1 FROM EDW_APPS.MATCHING.TEMP_MZK_TITLE_VARIANTS var
#         WHERE var.ORIGINAL_TITLE = orig.TITLE
#         AND var.IS_VARIANT = TRUE
#     )
#     """).collect()
    
#     # Create the final table
#     session.sql("""
#     CREATE OR REPLACE TABLE EDW_APPS.MATCHING.MZK_TRACKS_TITLE_VARIANTS AS
#     SELECT * FROM EDW_APPS.MATCHING.TEMP_MZK_TITLE_VARIANTS
#     //WHERE TITLE_VARIANT IS NOT NULL
#     """).collect()
    
#     # Get statistics
#     stats = session.sql("""
#     SELECT
#         SUM(CASE WHEN IS_VARIANT = FALSE THEN 1 ELSE 0 END) as ORIGINAL_COUNT,
#         SUM(CASE WHEN IS_VARIANT = TRUE THEN 1 ELSE 0 END) as VARIANT_COUNT,
#         COUNT(*) as TOTAL_COUNT
#     FROM EDW_APPS.MATCHING.MZK_TRACKS_TITLE_VARIANTS
#     """).collect()
    
#     print(f"Original tracks: {stats[0]['ORIGINAL_COUNT']}")
#     print(f"Generated variants: {stats[0]['VARIANT_COUNT']}")
#     print(f"Total rows in output: {stats[0]['TOTAL_COUNT']}")
    
#     return


# def main(session):
#     # Get the input data from MAZOOKA TRACKS
#     print("Reading data from MAZOOKA...")
#     table_name = "EDW_APPS.MATCHING.TRACKS"
    
#     print("\nSample data (5 rows):")
#     session.table(table_name).limit(5).show()
    
    
#     # Register JavaScript UDFs for title variant generation
#     register_udfs(session)
    
#     # Process data in SQL batches
#     process_batches_sql(session, table_name)
    
#     return session.table("EDW_APPS.MATCHING.MZK_TRACKS_TITLE_VARIANTS")

# # Run the main function
# result_df = main(session)

In [ ]:
# import snowflake.snowpark as snowpark
# from snowflake.snowpark.functions import col, lit, udf, call_udf, expr
# import pandas as pd
# import numpy as np
# import time

# # Define view names and result table
# ADC_WORKS_VIEW = "ADC_WORKS_TITLE_VARIANTS"
# MAZOOKA_TRACKS_VIEW = "MZK_TRACKS_TITLE_VARIANTS" 
# RESULTS_TABLE = "EDW_APPS.MATCHING.TITLE_VARIANT_MATCHING_RESULTS"

# # Batch processing parameters
# BATCH_SIZE = 50000
# PROGRESS_INTERVAL = 150000  # Report progress every 50000 rows

# def create_jaro_winkler_similarity_udf():
#     """Create JavaScript UDF for title similarity calculation using Jaro-Winkler distance"""
#     try:
#         print("Creating title similarity function with Jaro-Winkler algorithm...")
        
#         # Create the JavaScript UDF using dollar-quoting to avoid escape issues
#         title_similarity_js = """
#         CREATE OR REPLACE FUNCTION TITLE_SIMILARITY(str1 STRING, str2 STRING)
#         RETURNS FLOAT
#         LANGUAGE JAVASCRIPT
#         AS 
#         $$
#           // Normalize inputs - uppercase, remove extra spaces, special chars, etc.
#           function normalizeString(str) {
#             if (!str) return '';
#             return str.toUpperCase()
#                       .replace(/[^A-Z0-9\\s]/g, ' ')  // Replace special chars with space
#                       .replace(/\\s+/g, ' ')          // Replace multiple spaces with single space
#                       .trim();
#           }
          
#           // Jaro similarity implementation
#           function jaroSimilarity(s1, s2) {
#             // If the strings are equal
#             if (s1 === s2) return 1.0;
            
#             // If either string is empty
#             if (s1.length === 0 || s2.length === 0) return 0.0;
            
#             // Maximum distance allowed for matching
#             const matchDistance = Math.floor(Math.max(s1.length, s2.length) / 2) - 1;
            
#             // Arrays to track matches
#             const s1Matches = Array(s1.length).fill(false);
#             const s2Matches = Array(s2.length).fill(false);
            
#             // Count of matches
#             let matches = 0;
            
#             // Look for matches
#             for (let i = 0; i < s1.length; i++) {
#               // Lower and upper bound for matching
#               const start = Math.max(0, i - matchDistance);
#               const end = Math.min(i + matchDistance + 1, s2.length);
              
#               for (let j = start; j < end; j++) {
#                 // Skip if already matched or not matching
#                 if (s2Matches[j] || s1[i] !== s2[j]) continue;
                
#                 // Found a match
#                 s1Matches[i] = true;
#                 s2Matches[j] = true;
#                 matches++;
#                 break;
#               }
#             }
            
#             // If no matches, return 0
#             if (matches === 0) return 0.0;
            
#             // Count transpositions
#             let transpositions = 0;
#             let k = 0;
            
#             for (let i = 0; i < s1.length; i++) {
#               if (!s1Matches[i]) continue;
              
#               while (!s2Matches[k]) k++;
              
#               if (s1[i] !== s2[k]) transpositions++;
              
#               k++;
#             }
            
#             // Calculate Jaro similarity
#             const jaroSim = (
#               (matches / s1.length) +
#               (matches / s2.length) +
#               ((matches - transpositions / 2) / matches)
#             ) / 3;
            
#             return jaroSim;
#           }
          
#           // Jaro-Winkler similarity
#           function jaroWinklerSimilarity(s1, s2) {
#             const normalized1 = normalizeString(s1);
#             const normalized2 = normalizeString(s2);
            
#             // Calculate Jaro similarity
#             const jaroSim = jaroSimilarity(normalized1, normalized2);
            
#             // Calculate prefix length (max 4)
#             let prefixLength = 0;
#             const maxPrefixLength = Math.min(4, Math.min(normalized1.length, normalized2.length));
            
#             for (let i = 0; i < maxPrefixLength; i++) {
#               if (normalized1[i] === normalized2[i]) {
#                 prefixLength++;
#               } else {
#                 break;
#               }
#             }
            
#             // Scaling factor for how much the score is adjusted by prefix length
#             const scalingFactor = 0.1;
            
#             // Calculate Jaro-Winkler similarity
#             return jaroSim + (prefixLength * scalingFactor * (1 - jaroSim));
#           }
          
#           // Main entry point for the UDF
#           return jaroWinklerSimilarity(STR1, STR2);
#         $$
#         """
        
#         # Execute the SQL statement
#         session.sql(title_similarity_js).collect()
#         print("Title similarity UDF with Jaro-Winkler distance created successfully.")
#         return True
#     except Exception as e:
#         print(f"Error creating title similarity UDF: {e}")
#         return False        
        
# def check_table_exists(table_name):
#     """Check if a table exists"""
#     try:
#         result = session.sql(f"SELECT 1 FROM {table_name} LIMIT 1").collect()
#         print(f"Table {table_name} exists and is accessible.")
#         return True
#     except Exception as e:
#         print(f"Table {table_name} does not exist or is not accessible: {e}")
#         return False
        
# def prepare_title_variant_views():
#     """Create optimized enhanced views of title variants from ADC and Mazooka tables"""
#     try:
#         # Create enhanced ADC Works view with normalized titles - all uppercase
#         adc_view_sql = f"""
#         CREATE OR REPLACE TEMPORARY VIEW ADC_VARIANTS AS
#         SELECT
#             APRA_WORK_ID,
#             UPPER(TITLE) AS TITLE,
#             UPPER(TITLE_VARIANT) AS TITLE_VARIANT,
#             UPPER(ORIGINAL_TITLE) AS ORIGINAL_TITLE,
#             UPPER(ISWC) AS ISWC,
#             IS_VARIANT,
#             UPPER(TITLE) AS UPPER_TITLE,
#             UPPER(TITLE_VARIANT) AS UPPER_TITLE_VARIANT,
#             UPPER(ORIGINAL_TITLE) AS UPPER_ORIGINAL_TITLE,
#             UPPER(REGEXP_REPLACE(TITLE, '[^A-Z0-9\\\\s]', ' ')) AS CLEAN_TITLE,
#             UPPER(REGEXP_REPLACE(TITLE, '[^A-Z0-9]', '')) AS COMPACT_TITLE,
#             UPPER(REGEXP_REPLACE(REGEXP_REPLACE(TITLE, '[^A-Z0-9\\\\s]', ' '), '^(THE|A|AN)\\\\s+', '')) AS NO_ARTICLE_TITLE
#         FROM {ADC_WORKS_VIEW}
#         """
        
#         # Create enhanced Mazooka Tracks view with normalized titles - all uppercase
#         tracks_view_sql = f"""
#         CREATE OR REPLACE TEMPORARY VIEW TRACKS_VARIANTS AS
#         SELECT
#             TRACK_ID,
#             UPPER(TITLE) AS TITLE,
#             UPPER(TITLE_VARIANT) AS TITLE_VARIANT,
#             UPPER(ORIGINAL_TITLE) AS ORIGINAL_TITLE,
#             UPPER(ISWC) AS ISWC,
#             IS_VARIANT,
#             UPPER(TITLE) AS UPPER_TITLE,
#             UPPER(TITLE_VARIANT) AS UPPER_TITLE_VARIANT,
#             UPPER(ORIGINAL_TITLE) AS UPPER_ORIGINAL_TITLE,
#             UPPER(REGEXP_REPLACE(TITLE, '[^A-Z0-9\\\\s]', ' ')) AS CLEAN_TITLE,
#             UPPER(REGEXP_REPLACE(TITLE, '[^A-Z0-9]', '')) AS COMPACT_TITLE,
#             UPPER(REGEXP_REPLACE(REGEXP_REPLACE(TITLE, '[^A-Z0-9\\\\s]', ' '), '^(THE|A|AN)\\\\s+', '')) AS NO_ARTICLE_TITLE
#         FROM {MAZOOKA_TRACKS_VIEW}
#         """
        
#         # Execute the SQL statements
#         session.sql(adc_view_sql).collect()
#         session.sql(tracks_view_sql).collect()
        
#         print("Created enhanced views with normalized uppercase titles")
#         return True
#     except Exception as e:
#         print(f"Error creating enhanced views: {e}")
#         return False

# def create_initial_results_table():
#     """Create the results table structure to store matches"""
#     try:
#         create_table_sql = f"""
#         CREATE OR REPLACE TABLE {RESULTS_TABLE} (
#             APRA_WORK_ID VARCHAR,
#             APRA_TITLE VARCHAR,
#             MUZOOKA_TRACK_ID VARCHAR,
#             MUZOOKA_TITLE VARCHAR,
#             TITLE_MATCH_SCORE FLOAT,
#             APRA_ISWC VARCHAR,
#             MUZOOKA_ISWC VARCHAR
#         )
#         """
#         session.sql(create_table_sql).collect()
#         print(f"Created results table: {RESULTS_TABLE}")
#         return True
#     except Exception as e:
#         print(f"Error creating results table: {e}")
#         return False

# def get_total_source_counts():
#     """Get total counts of records to process for progress reporting"""
#     try:
#         adc_count = session.sql("SELECT COUNT(*) as count FROM ADC_VARIANTS").collect()[0]['COUNT']
#         tracks_count = session.sql("SELECT COUNT(*) as count FROM TRACKS_VARIANTS").collect()[0]['COUNT']
#         print(f"Total ADC Works records: {adc_count}")
#         print(f"Total Muzooka Tracks records: {tracks_count}")
#         return adc_count, tracks_count
#     except Exception as e:
#         print(f"Error getting record counts: {e}")
#         return 0, 0
        
# def calculate_title_match_scores_batch():
#     """Calculate match scores between title variants using batched processing"""
#     try:
#         # Create the results table
#         if not create_initial_results_table():
#             return False
            
#         # Get total counts for progress reporting
#         adc_count, tracks_count = get_total_source_counts()
        
#         # Create a temporary table for batched results
#         session.sql("""
#         CREATE OR REPLACE TEMPORARY TABLE BATCH_MATCHES (
#             APRA_WORK_ID VARCHAR,
#             APRA_TITLE VARCHAR,
#             MUZOOKA_TRACK_ID VARCHAR,
#             MUZOOKA_TITLE VARCHAR,
#             TITLE_MATCH_SCORE FLOAT,
#             APRA_ISWC VARCHAR,
#             MUZOOKA_ISWC VARCHAR
#         )
#         """).collect()
        
#         # Create a temporary table for the current batch
#         session.sql("""
#         CREATE OR REPLACE TEMPORARY TABLE CURRENT_BATCH (
#             APRA_WORK_ID VARCHAR,
#             TITLE VARCHAR,
#             TITLE_VARIANT VARCHAR,
#             ORIGINAL_TITLE VARCHAR,
#             ISWC VARCHAR,
#             IS_VARIANT BOOLEAN,
#             UPPER_TITLE VARCHAR,
#             UPPER_TITLE_VARIANT VARCHAR,
#             UPPER_ORIGINAL_TITLE VARCHAR,
#             CLEAN_TITLE VARCHAR,
#             COMPACT_TITLE VARCHAR,
#             NO_ARTICLE_TITLE VARCHAR
#         )
#         """).collect()
        
#         # Process ADC records in batches
#         total_processed = 0
#         start_time = time.time()
        
#         # Get the total number of batches
#         total_batches = -(-adc_count // BATCH_SIZE)  # Ceiling division
        
#         for batch_num in range(total_batches):
#             batch_start_time = time.time()
#             batch_start = batch_num * BATCH_SIZE
            
#             # Clear the current batch table and populate with new batch
#             session.sql("TRUNCATE TABLE CURRENT_BATCH").collect()
            
#             # Load current batch into the temporary table
#             batch_load_sql = f"""
#             INSERT INTO CURRENT_BATCH
#             SELECT *
#             FROM ADC_VARIANTS
#             ORDER BY APRA_WORK_ID
#             LIMIT {BATCH_SIZE} OFFSET {batch_start}
#             """
#             session.sql(batch_load_sql).collect()
            
#             # Count actual records in this batch for progress reporting
#             batch_count_result = session.sql("SELECT COUNT(*) as count FROM CURRENT_BATCH").collect()
#             records_in_batch = batch_count_result[0]['COUNT']
            
#             # Skip if empty batch
#             if records_in_batch == 0:
#                 break
                
#             # Process matching for this batch using the temporary table
#             batch_match_sql = """
#             INSERT INTO BATCH_MATCHES
#             SELECT 
#                 a.APRA_WORK_ID,
#                 a.TITLE AS APRA_TITLE,
#                 t.TRACK_ID AS MUZOOKA_TRACK_ID,
#                 t.TITLE AS MUZOOKA_TITLE,
#                 GREATEST(
#                     -- Calculate exact match scores
#                     CASE 
#                         WHEN a.UPPER_TITLE = t.UPPER_TITLE THEN 1.0
#                         WHEN a.CLEAN_TITLE = t.CLEAN_TITLE THEN 0.95
#                         WHEN a.COMPACT_TITLE = t.COMPACT_TITLE THEN 0.90
#                         WHEN a.NO_ARTICLE_TITLE = t.NO_ARTICLE_TITLE THEN 0.85
#                         ELSE 0.0
#                     END,
#                     -- Calculate fuzzy match score using Jaro-Winkler
#                     TITLE_SIMILARITY(a.TITLE, t.TITLE)
#                 ) AS TITLE_MATCH_SCORE,
#                 a.ISWC AS APRA_ISWC,
#                 t.ISWC AS MUZOOKA_ISWC
#             FROM 
#                 CURRENT_BATCH a
#             CROSS JOIN 
#                 TRACKS_VARIANTS t
#             WHERE
#                 -- Filter combinations for better performance
#                 a.UPPER_TITLE = t.UPPER_TITLE OR
#                 a.CLEAN_TITLE = t.CLEAN_TITLE OR
#                 a.COMPACT_TITLE = t.COMPACT_TITLE OR
#                 a.NO_ARTICLE_TITLE = t.NO_ARTICLE_TITLE OR
#                 TITLE_SIMILARITY(a.TITLE, t.TITLE) >= 0.8
#             """
            
#             # Execute the batch processing
#             session.sql(batch_match_sql).collect()
            
#             # Update the processed count
#             total_processed += records_in_batch
            
#             # Get batch match count for reporting
#             batch_matches = session.sql("""
#                 SELECT COUNT(*) as count FROM BATCH_MATCHES 
#                 WHERE APRA_WORK_ID IN (SELECT APRA_WORK_ID FROM CURRENT_BATCH)
#             """).collect()[0]['COUNT']
            
#             # Report progress at each interval or for the batch
#             if total_processed % PROGRESS_INTERVAL == 0 or batch_num == total_batches - 1:
#                 elapsed_time = time.time() - start_time
#                 batch_time = time.time() - batch_start_time
#                 percent_complete = (total_processed / adc_count) * 100
#                 estimated_total_time = elapsed_time / percent_complete * 100 if percent_complete > 0 else 0
#                 estimated_remaining = estimated_total_time - elapsed_time
                
#                 print(f"Progress: {total_processed:,}/{adc_count:,} rows processed ({percent_complete:.2f}%)")
#                 print(f"Batch {batch_num + 1}/{total_batches} completed in {batch_time:.2f} seconds")
#                 print(f"Batch size: {records_in_batch:,} records, Found {batch_matches:,} matches")
#                 print(f"Elapsed time: {elapsed_time:.2f} seconds")
#                 print(f"Estimated remaining time: {estimated_remaining:.2f} seconds")
#                 print(f"Estimated total time: {estimated_total_time:.2f} seconds")
#                 print("-" * 50)
        
#         # Once all batches are processed, move results to the final table
#         print("Processing complete. Moving matches to final results table...")
#         insert_final_sql = f"""
#         INSERT INTO {RESULTS_TABLE}
#         SELECT * FROM BATCH_MATCHES
#         WHERE TITLE_MATCH_SCORE >= 0.8
#         ORDER BY TITLE_MATCH_SCORE DESC
#         """
        
#         session.sql(insert_final_sql).collect()
#         final_count = session.sql(f"SELECT COUNT(*) as count FROM {RESULTS_TABLE}").collect()[0]['COUNT']
        
#         print(f"All batches processed. Total ADC records processed: {total_processed:,}")
#         print(f"Final match count in {RESULTS_TABLE}: {final_count:,}")
        
#         return True
#     except Exception as e:
#         print(f"Error in batch processing: {e}")
#         return False

# def generate_match_statistics():
#     """Generate statistics about the matches"""
#     try:
#         stats_sql = f"""
#         SELECT 
#             COUNT(*) AS TOTAL_MATCHES,
#             COUNT(DISTINCT APRA_WORK_ID) AS UNIQUE_WORKS,
#             COUNT(DISTINCT MUZOOKA_TRACK_ID) AS UNIQUE_TRACKS,
            
#             -- Match score statistics
#             SUM(CASE WHEN TITLE_MATCH_SCORE = 1.0 THEN 1 ELSE 0 END) AS PERFECT_MATCHES,
#             SUM(CASE WHEN TITLE_MATCH_SCORE >= 0.9 AND TITLE_MATCH_SCORE < 1.0 THEN 1 ELSE 0 END) AS HIGH_MATCHES,
#             SUM(CASE WHEN TITLE_MATCH_SCORE >= 0.8 AND TITLE_MATCH_SCORE < 0.9 THEN 1 ELSE 0 END) AS MED_HIGH_MATCHES,
#             AVG(TITLE_MATCH_SCORE) AS AVG_MATCH_SCORE,
            
#             -- ISWC statistics
#             SUM(CASE WHEN APRA_ISWC = MUZOOKA_ISWC AND APRA_ISWC IS NOT NULL AND APRA_ISWC != '' THEN 1 ELSE 0 END) AS MATCHING_ISWC_COUNT
#         FROM 
#             {RESULTS_TABLE}
#         """
        
#         stats = session.sql(stats_sql).to_pandas()
#         print("\nMatch Statistics:")
#         print(stats)
#         return stats
#     except Exception as e:
#         print(f"Error generating match statistics: {e}")
#         return None

# def get_example_matches(limit=10):
#     """Get example matches for verification"""
#     try:
#         examples_sql = f"""
#         SELECT 
#             APRA_WORK_ID,
#             APRA_TITLE,
#             MUZOOKA_TRACK_ID,
#             MUZOOKA_TITLE,
#             TITLE_MATCH_SCORE,
#             APRA_ISWC,
#             MUZOOKA_ISWC
#         FROM 
#             {RESULTS_TABLE}
#         ORDER BY 
#             TITLE_MATCH_SCORE DESC
#         LIMIT {limit}
#         """
        
#         examples = session.sql(examples_sql).to_pandas()
#         print("\nExample Matches:")
#         print(examples)
#         return examples
#     except Exception as e:
#         print(f"Error getting example matches: {e}")
#         return None

# def run_title_variant_matching():
#     """Main function to execute title variant matching with batch processing"""
#     print("Starting title variant matching between ADC Works and Mazooka Tracks with batch processing...")
#     print(f"Batch size: {BATCH_SIZE} rows, Progress reporting every {PROGRESS_INTERVAL} rows")
    
#     start_time = time.time()
    
#     # Step 1: Create Jaro-Winkler similarity UDF
#     if not create_jaro_winkler_similarity_udf():
#         print("Failed to create Jaro-Winkler similarity UDF. Aborting process.")
#         return False
    
#     # Step 2: Check if required views exist
#     adc_view_exists = check_table_exists(ADC_WORKS_VIEW)
#     tracks_view_exists = check_table_exists(MAZOOKA_TRACKS_VIEW)
    
#     if not (adc_view_exists and tracks_view_exists):
#         missing = []
#         if not adc_view_exists: missing.append(ADC_WORKS_VIEW)
#         if not tracks_view_exists: missing.append(MAZOOKA_TRACKS_VIEW)
        
#         print(f"The following required objects are missing: {', '.join(missing)}")
#         print("Please ensure all required objects exist. Aborting process.")
#         return False
    
#     # Step 3: Prepare enhanced views with normalized titles
#     if not prepare_title_variant_views():
#         print("Failed to prepare enhanced views. Aborting process.")
#         return False
    
#     # Step 4: Calculate title match scores using batch processing
#     if not calculate_title_match_scores_batch():
#         print("Failed to calculate title match scores. Aborting process.")
#         return False
    
#     # Generate match statistics and examples
#     generate_match_statistics()
#     get_example_matches(10)
    
#     total_time = time.time() - start_time
#     print(f"\nTitle variant matching process completed successfully in {total_time:.2f} seconds!")
#     print(f"Results saved to table: {RESULTS_TABLE}")
    
#     return True

# if __name__ == "__main__":
#     try:
#         success = run_title_variant_matching()
#         if success:
#             print("\nTitle variant matching with batch processing completed successfully.")
#             print("Check the results table for matched titles.")
#         else:
#             print("\nTitle variant matching failed. See error messages above.")
#     except Exception as e:
#         print(f"\nUnexpected error in title variant matching: {e}")